# Analysis with five probes excluded
Excluded before all calculations: n-butylamine, morpholine, piperidine, phosphorus tribromide and thionyl chloride. The source Excel is unchanged. Restart the kernel and Run All to regenerate plots, tables, correlations, clustering and permutation tests. Old outputs have been cleared.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import scipy.cluster.hierarchy as sch
import seaborn as sns

from IPython.display import display
from joblib import Parallel, delayed
from scipy.linalg import eigh
from scipy.spatial.distance import pdist
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans, SpectralClustering
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.model_selection import GridSearchCV, KFold, LeaveOneOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

sns.set()
plt.rcParams['figure.dpi'] = 600
plt.rcParams['savefig.dpi'] = 600


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)


# Project overview

We cluster solvents and sorbates to make sense of how solvent properties relate to graphene oxide sorption. The goal is practical: find groups of substances that behave similarly so we can link simple measured properties to observed adsorption patterns and guide further experiments and spectra analysis.

Why several clustering methods? Real experimental data are messy and each algorithm reveals a different facet. K-Means gives a quick, easy partition for balanced, spherical groups. Hierarchical clustering shows nested relationships and helps pick natural cut levels. DBSCAN finds dense, irregular groups and isolates noise. Spectral clustering can detect complex shapes but is sensitive to the similarity scale. Running several methods helps check which groupings are stable and which are artefacts of the algorithm.

What we deliver
- concise tables with cluster assignments for each method, so you can compare results side by side;  
- linear correlation observation to confirm the hypothesis of chemical nature of solvent interactions with graphene oxide;
- Plotly dashboards to inspect individual points and test hypotheses;  

Short rationale for choices
- We prioritize interpretability and physical plausibility over raw numeric scores. If a method gives clusters that match known chemistry, we trust it more.  
- We use multiple metrics to support the final choice, but human judgement and domain knowledge are decisive.  
- Because the data are noisy, we expect imperfect clusters. The goal is to extract robust, actionable patterns rather than perfect partitioning.



## 1. Data Loading & Analysis
##### Clustering was based on features such as the permittivity and sorption of the solvents listed below on graphene oxides. In the future, new, purely chemical parameters will be used for multi-feature clustering. The final dataset will be included in the project files.

In [ ]:
df = pd.read_excel('Graphene.xlsx', sheet_name='Epsilon')
df.columns = df.columns.astype(str).str.strip()

# Apply the same exclusions before ALL correlations and clustering.
# Match exact normalized names, never substrings (e.g. pyrrolidine is retained).
import unicodedata

def _probe_key(value):
    text = unicodedata.normalize('NFKC', str(value)).casefold()
    return ''.join(ch for ch in text if ch.isalnum())

EXCLUDED_PROBE_ALIASES = {
    'n-Butylamine': ['n-Butylamine', '1-Butylamine', 'Butylamine',
                    'Butan-1-amine', '1-Aminobutane', '1-бутиламин', 'н-бутиламин'],
    'Morpholine': ['Morpholine', 'Морфолин'],
    'Piperidine': ['Piperidine', 'Пиперидин'],
    'Phosphorus tribromide': ['Phosphorus tribromide', 'PBr3',
                            'Трибромид фосфора', 'Phosphorus(III) bromide'],
    'Thionyl chloride': ['Thionyl chloride', 'SOCl2', 'Тионилхлорид', 'Хлористый тионил'],
}
_excluded_keys = {_probe_key(alias) for aliases in EXCLUDED_PROBE_ALIASES.values()
                  for alias in aliases}
_exclude_mask = df['Name'].map(_probe_key).isin(_excluded_keys)
excluded_probes = df.loc[_exclude_mask].copy()
df = df.loc[~_exclude_mask].copy().reset_index(drop=True)
assert not df['Name'].map(_probe_key).isin(_excluded_keys).any()
print(f'Excluded {len(excluded_probes)} rows; retained {len(df)} rows for analysis.')
display(excluded_probes[['Name']])
_found_keys = set(excluded_probes['Name'].map(_probe_key))
_missing = [name for name, aliases in EXCLUDED_PROBE_ALIASES.items()
            if not any(_probe_key(alias) in _found_keys for alias in aliases)]
if _missing:
    print('Not found in this Excel input (check names):', ', '.join(_missing))
df.head()


In [ ]:
df.info()


In [ ]:
df['Number of substance'] = df['Name'].factorize()[0]
df.sample(5)


In [ ]:
plt.figure(figsize=(10,8))
plt.scatter(df['Sorption, mmol/g'], df['ε'], s = 35, edgecolor = 'k')
plt.xlabel('Sorption, mmol/g', fontsize=14)
plt.ylabel('ε (Dielectric constant)', fontsize=14)
plt.title('Sorption vs Dielectric constant data', fontsize=14)


## 2. Baseline models: K-Means and Agglomerative Hierarchical Clustering

K-Means is a fast, partition-based algorithm that divides the dataset into $K$ clusters by minimizing the within-cluster sum of squared distances; it works well for roughly spherical, similarly sized clusters but can be sensitive to initialization and cluster shape. Agglomerative hierarchical clustering is a bottom-up procedure that successively merges the most similar clusters to produce a tree; it reveals nested structure and does not require specifying $K$ a priori, but results depend on the chosen linkage and distance metric.

To select the number of clusters we used two complementary tools. The **Elbow method** evaluates the within-cluster sum of squares as a function of $K$.

The value of $K$ at which the inertia curve shows a clear “elbow” is taken as a parsimonious choice balancing fit and complexity. The **dendrogram** visualizes cluster merges and merging distances — a large vertical jump in the dendrogram indicates a natural split and suggests an appropriate cut level. Together, the elbow plot provides a quantitative guide for partitioning methods while the dendrogram supplies qualitative insight into hierarchical structure; using them in tandem improves robustness of the cluster-number decision.


### 2.1. Recognizing the optimal number of clusters

In [ ]:
X = df[['Sorption, mmol/g', 'ε']].apply(pd.to_numeric, errors='coerce').to_numpy(float)
X_scaled = StandardScaler().fit_transform(X)

plt.figure(figsize=(10, 8))
wcss = []
for k in range(1, 11):
    model = KMeans(n_clusters=k, init='k-means++', n_init=50, random_state=42)
    model.fit(X_scaled)
    wcss.append(model.inertia_)
sns.pointplot(x=range(1, 11), y=wcss)
plt.title('The Elbow curve')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()


In [ ]:
hierarchical_candidates = []
for linkage_name in ('ward', 'complete', 'average'):
    for k in range(2, 9):
        candidate = AgglomerativeClustering(n_clusters=k, linkage=linkage_name)
        labels = candidate.fit_predict(X_scaled)
        hierarchical_candidates.append((
            silhouette_score(X_scaled, labels),
            -davies_bouldin_score(X_scaled, labels),
            k,
            linkage_name
        ))

_, _, optimal_hierarchical_k, optimal_linkage = max(hierarchical_candidates)

df_removal = df.drop_duplicates(subset=['Name'], keep='last').copy()
X_dendrogram = StandardScaler().fit_transform(df_removal[['Sorption, mmol/g', 'ε']])
Z = sch.linkage(X_dendrogram, method=optimal_linkage)

patterns = {
    'oxygenated': r'Dimethylformamide|N-Methyl-2-pyrrolidone|Citraconic anhydride|Furfural|Acetylacetone|Acetic anhydride|Water|Heavy water|Hydrogen peroxide|Ethyl acetate|Acetic acid|Tetrahydrofuran|Dibutyl phthalate|1,4-Dioxane',
    'nitrogenated': r'N-Methyl-2-pyrrolidone|Acetonitrile|Dimethylformamide|Pyridine|Pyrrolidine|n-Butylamine|Dicyclohexylamine|N,N,N\',N\'-Tetramethylethylenediamine|Pyridine-d5|Morpholine|Pyridine|N-Methylaniline|Pyridazine|Et3N|Triethylenetetramine|Pyridine|Aniline|Piperidine|Morpholine|N,N-Dimethylformamide|Diethylamine|Pivalonitrile|Benzonitrile',
    'sulfur': r'Dimethyl sulfoxide|Thionyl chloride|Thiophos',
    'phosphorus': r'Tri-n-butyl phosphate|Phosphorus tribromide'
}

def auto_group(name):
    for group, pattern in patterns.items():
        if re.search(pattern, name):
            return group
    return 'other'

color_for_group = {
    'oxygenated': 'red',
    'nitrogenated': 'blue',
    'sulfur': '#FFA600',
    'phosphorus': '#F311EF',
    'other': 'black'
}

if 1 < optimal_hierarchical_k < len(df_removal):
    lower_distance = Z[-optimal_hierarchical_k, 2]
    upper_distance = Z[-optimal_hierarchical_k + 1, 2]
    color_threshold = (lower_distance + upper_distance) / 2
else:
    color_threshold = None

fig = plt.figure(figsize=(12, 10), dpi=600)
sch.dendrogram(
    Z,
    labels=df_removal['Name'].astype(str).to_numpy(),
    orientation='left',
    leaf_font_size=10,
    color_threshold=color_threshold
)
current_ax = plt.gca()
for label in current_ax.get_yticklabels():
    label.set_color(color_for_group[auto_group(label.get_text())])
plt.title('Dendrogram')
plt.xlabel('Euclidean distances')
plt.tight_layout()
plt.show()
print(f'Optimal hierarchical clustering: k={optimal_hierarchical_k}, linkage={optimal_linkage}')


### 2.2. K-Means clustering

In [ ]:
kmeans_candidates = []
for k in range(2, 9):
    candidate = KMeans(n_clusters=k, init='k-means++', n_init=50, random_state=42)
    labels = candidate.fit_predict(X_scaled)
    kmeans_candidates.append((silhouette_score(X_scaled, labels), -davies_bouldin_score(X_scaled, labels), k))

_, _, optimal_kmeans_k = max(kmeans_candidates)
kmeans = KMeans(n_clusters=optimal_kmeans_k, init='k-means++', n_init=50, random_state=42)
y_kmeans = kmeans.fit_predict(X_scaled)
print(f'Optimal K-Means clusters: {optimal_kmeans_k}')


In [ ]:
plt.figure(figsize=(8, 6))
for cluster in sorted(np.unique(y_kmeans)):
    mask = y_kmeans == cluster
    plt.scatter(X[mask, 0], X[mask, 1], s=45, label=f'Cluster {cluster + 1}', edgecolors='black')

centers = StandardScaler().fit(X).inverse_transform(kmeans.cluster_centers_)
plt.scatter(centers[:, 0], centers[:, 1], s=90, label='Centroids', c='black', edgecolors='yellow', marker='X')
plt.title('Clusters of substances (K-Means)')
plt.xlabel('Sorption (mmol/g)')
plt.ylabel('ε (Dielectric constant)')
plt.legend()
plt.show()


In [ ]:
def cluster_df(labels):
    labels = np.asarray(labels)
    names = np.where(labels == -1, 'Noise', (labels + 1).astype(str))
    return pd.DataFrame({
        'Name': df['Name'],
        'Cluster': names,
        'ε (Dielectric constant)': df['ε'].round(2),
        'Sorption (mmol/g)': df['Sorption, mmol/g'].round(3)
    })


In [ ]:
df_kmeans = cluster_df(y_kmeans).sort_values('Cluster').reset_index(drop=True)
df_kmeans


### 2.3. Hierarchical clustering

In [ ]:
hc = AgglomerativeClustering(n_clusters=optimal_hierarchical_k, linkage=optimal_linkage)
y_hc = hc.fit_predict(X_scaled)
print(f'Optimal hierarchical clustering: k={optimal_hierarchical_k}, linkage={optimal_linkage}')


In [ ]:
plt.figure(figsize=(8, 6))
for cluster in sorted(np.unique(y_hc)):
    mask = y_hc == cluster
    plt.scatter(X[mask, 0], X[mask, 1], s=45, label=f'Cluster {cluster + 1}', edgecolors='black')
plt.title('Clusters of substances (Hierarchical)')
plt.xlabel('Sorption (mmol/g)')
plt.ylabel('ε (Dielectric constant)')
plt.legend()
plt.show()


In [ ]:
df_hc = cluster_df(y_hc).sort_values('Cluster').reset_index(drop=True)
df_hc


## 3. Spectral Clustering procedure

Spectral clustering represents the dataset as a weighted graph and finds clusters by analysing the spectrum of a graph Laplacian. The main steps used in this work are:
 * Construct an adjacency matrix $A$ from pairwise distances
 * Form the degree matrix $D$ and a Laplacian $L$ or a normalized variant
 * Compute eigenvalues and eigenvectors of $L$
 * Select $K$ using the eigengap heuristic and embed the data into the subspace spanned by the first $K$ eigenvectors
 * Run a standard partitioning algorithm (e.g. K-Means) in that embedding.


### 3.1. Adjacency matrix

We build the adjacency matrix $A\in\mathbb{R}^{n\times n}$ using a Gaussian kernel:

$
A_{ij} = \exp\!\left( -\frac{\lVert x_i - x_j \rVert^2}{2\sigma^2} \right)
$


where $x_i,x_j\in\mathbb{R}^p$ are feature vectors for samples $i$ and $j$, and $\sigma>0$ controls the kernel width (similarity scale). Practical notes:
- $x_i,x_j$ are the vectors of chosen features.
- $\sigma$ sets the distance scale: small $\sigma$ → only very close points get large weight; large $\sigma$ → many weights near 1.
- Common heuristics: choose $\sigma$ as the median of pairwise distances, or tune it by inspecting the adjacency distribution.


In [ ]:
distance_matrix = pdist(X_scaled, metric='euclidean')
sigma = np.median(distance_matrix)
A = -np.square(X_scaled[:, None] - X_scaled[None, :]).sum(axis=-1)
A = np.exp(A / (2 * sigma ** 2))
np.fill_diagonal(A, 0)
A


### 3.2 Adjacency value distribution plot

The histogram of $A_{ij}$ values helps to diagnose the affinity construction:
- a large spike near zero means the affinity matrix is **sparse**;
- a spread of small but nonzero values suggests intermediate similarities and sensitivity to $\sigma$;
- multimodality or heavy tails indicate heterogeneous local densities (clusters with differing internal scales).

Use this plot to adjust $\sigma$. If almost all values are ≈0 or ≈1, the kernel width should be re-tuned.

Note: we've tried to take the common which is $\sigma$ = 1. In the bottom of the particular notebook you will be able to see the adjustment of the $\sigma$-value


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

bin_values, _, _ = ax1.hist(A.flatten(), bins=50)

ax2.hist(A.flatten(), bins=50)
ax2.set_ylim([0, bin_values[9]])
ax1.set_xlabel('Value')
ax2.set_xlabel('Value')
ax1.set_ylabel('Frequency')
ax1.set_title('Adjacency Matrix Values')
ax2.set_title('Area of small values (first 50 bins)')
fig.suptitle('Adjacency Matrix Value Distribution')
plt.show()


In [ ]:
D = np.diag(np.sum(A, axis=1))
L= np.eye(A.shape[0]) - np.linalg.inv(np.sqrt(D)) @ A @ np.linalg.inv(np.sqrt(D))
eigenvalues, eigenvectors = eigh(L)

sorted_indices = np.argsort(eigenvalues)
eigenvalues_sorted = eigenvalues[sorted_indices]

plt.figure(figsize=(10, 6))
plt.plot(np.arange(0, min(20, len(eigenvalues_sorted))), 
            eigenvalues_sorted[1:21], 'bo-') 
plt.xticks(np.arange(0, min(20, len(eigenvalues_sorted)), 1))
plt.xlabel('The index of the eigenvalue')
plt.ylabel('Eigenvalue')
plt.title('Laplacian Eigenvalues')
plt.show()


In [ ]:
eigengap = np.diff(eigenvalues_sorted[1:11])
optimal_k_eigengap = np.argmax(eigengap) + 2
eigen_df = pd.DataFrame({
    'Difference between Eigenvalues': eigengap,
    'Number of clusters': np.arange(2, 11)
})
eigen_df.T


In [ ]:
spectral_candidates = []
for gamma in (0.25, 0.5, 1.0, 2.0, 4.0, 8.0):
    for k in range(2, 9):
        try:
            candidate = SpectralClustering(
                n_clusters=k,
                affinity='rbf',
                gamma=gamma,
                assign_labels='kmeans',
                n_init=30,
                random_state=42
            )
            labels = candidate.fit_predict(X_scaled)
            spectral_candidates.append((
                silhouette_score(X_scaled, labels),
                -davies_bouldin_score(X_scaled, labels),
                k,
                gamma
            ))
        except ValueError:
            pass

_, _, optimal_spectral_k, optimal_gamma = max(spectral_candidates)
spectral = SpectralClustering(
    n_clusters=optimal_spectral_k,
    affinity='rbf',
    gamma=optimal_gamma,
    assign_labels='kmeans',
    n_init=30,
    random_state=42
)
y_spectral = spectral.fit_predict(X_scaled)
print(f'Optimal spectral clustering: k={optimal_spectral_k}, gamma={optimal_gamma}')


In [ ]:
plt.figure(figsize=(8, 6))
for cluster in sorted(np.unique(y_spectral)):
    mask = y_spectral == cluster
    plt.scatter(X[mask, 0], X[mask, 1], s=45, label=f'Cluster {cluster + 1}', edgecolors='black')
plt.title('Clusters of substances (Spectral Clustering)')
plt.xlabel('Sorption (mmol/g)')
plt.ylabel('ε (Dielectric constant)')
plt.legend()
plt.show()


In [ ]:
df_spectral = cluster_df(y_spectral).sort_values('Cluster').reset_index(drop=True)
df_spectral


## 4. DBSCAN

DBSCAN is a density-based clustering method that groups together points that lie in dense regions of the data space. Instead of forcing every point into a cluster, it separates dense structures from scattered or noisy points. This makes the method useful when the data contain irregular shapes or many outliers.  
It requires two main parameters: the neighborhood radius and the minimum number of points that define a dense area. When these conditions are met, points expand into connected regions that form clusters. Everything that remains isolated is treated as noise.  
This approach helps reveal natural patterns in data without assuming how many clusters should exist beforehand.


In [ ]:
dbscan_candidates = []
for eps in np.round(np.linspace(0.15, 1.50, 28), 2):
    for min_samples in range(3, 9):
        candidate = DBSCAN(eps=float(eps), min_samples=min_samples)
        labels = candidate.fit_predict(X_scaled)
        mask = labels != -1
        unique = np.unique(labels[mask])
        if mask.sum() < 3 or len(unique) < 2 or (1 - mask.mean()) > 0.45:
            continue
        counts = pd.Series(labels[mask]).value_counts()
        if counts.min() < 2:
            continue
        silhouette = silhouette_score(X_scaled[mask], labels[mask])
        noise_fraction = 1 - mask.mean()
        dbscan_candidates.append((silhouette - 0.20 * noise_fraction, silhouette, -noise_fraction, eps, min_samples))

_, _, _, optimal_eps, optimal_min_samples = max(dbscan_candidates)
dbscan = DBSCAN(eps=float(optimal_eps), min_samples=int(optimal_min_samples)).fit(X_scaled)
y_dbscan = dbscan.labels_
print(f'Optimal DBSCAN: eps={optimal_eps}, min_samples={optimal_min_samples}')


In [ ]:
plt.figure(figsize=(8, 6))
for cluster in sorted(np.unique(y_dbscan)):
    mask = y_dbscan == cluster
    label = 'Noise' if cluster == -1 else f'Cluster {cluster + 1}'
    plt.scatter(X[mask, 0], X[mask, 1], s=45, label=label, edgecolors='black')
plt.legend()
plt.title('Clusters of substances (DBSCAN)')
plt.xlabel('Sorption (mmol/g)')
plt.ylabel('ε (Dielectric constant)')
plt.show()


In [ ]:
y_dbscan = dbscan.labels_
df_dbscan = cluster_df(y_dbscan).sort_values('Cluster').reset_index(drop=True)
df_dbscan


## Internal Cluster Validation

To compare the clustering methods, we used three internal validation metrics: **Silhouette**, **Calinski–Harabasz**, and **Davies–Bouldin**.  
These metrics help evaluate how well the clusters are formed without needing external labels.

### 1) Silhouette Score
Silhouette measures how similar a point is to its own cluster compared to other clusters.

$
s = \frac{b - a}{\max(a, b)}
$

- $a$: average distance to points in the same cluster  
- $b$: lowest average distance to points in other clusters  
- Values range from $-1$ to $1$. Higher is better.

### 2) Calinski–Harabasz Index
This score compares the dispersion between clusters to the dispersion inside clusters.

$
CH = \frac{SS_B/(k-1)}{SS_W/(n-k)}
$

- $SS_B$: between-cluster dispersion  
- $SS_W$: within-cluster dispersion  
- Higher values mean better clustering.

### 3) Davies–Bouldin Index
This index measures how well clusters are separated.

$
DB = \frac{1}{k} \sum_{i=1}^{k} \max_{j \neq i} 
\left( \frac{s_i + s_j}{d_{ij}} \right)
$

- $s_i$: average distance of points in cluster \(i\) to its center  
- $d_{ij}$: distance between cluster centers  
- Lower values are better.


In [ ]:
def clustering_metrics(method, labels, parameters, noise=False):
    labels = np.asarray(labels)
    mask = labels != -1 if noise else np.ones(len(labels), dtype=bool)
    return {
        'Method': method,
        'Parameters': parameters,
        'Silhouette': silhouette_score(X_scaled[mask], labels[mask]),
        'Calinski-Harabasz': calinski_harabasz_score(X_scaled[mask], labels[mask]),
        'Davies-Bouldin': davies_bouldin_score(X_scaled[mask], labels[mask]),
        'Noise_fraction': 1 - mask.mean()
    }

metrics_df = pd.DataFrame([
    clustering_metrics('K-Means', y_kmeans, f'k={optimal_kmeans_k}'),
    clustering_metrics('Hierarchical', y_hc, f'k={optimal_hierarchical_k}, linkage={optimal_linkage}'),
    clustering_metrics('Spectral', y_spectral, f'k={optimal_spectral_k}, gamma={optimal_gamma}'),
    clustering_metrics('DBSCAN', y_dbscan, f'eps={optimal_eps}, min_samples={optimal_min_samples}', noise=True)
])


In [ ]:
metrics_df.round(2)


## RANSAC regression with sorption as the response

For every one-dimensional regression, the molecular descriptor is used as $X$ and the experimentally measured sorption is used as $Y$. RANSAC therefore evaluates vertical residuals in sorption units (mmol/g).

The residual threshold $\tau$ is not fixed at one and is not generated with excessive decimal precision. For every fitted line it is selected from the transparent grid:

$$
\tau \in \{0.25,\ 0.50,\ 0.75,\ 1.00,\ 1.25,\ 1.50,\ 2.00,\ 2.50,\ 3.00\}\ \mathrm{mmol/g}.
$$

Candidate settings are compared using leave-one-out $Q^2$ multiplied by inlier coverage. The selected $\tau$ is reported exactly as a grid value, without false precision such as 1.064.


After the final least-squares refit, the inlier mask is recalculated from the final vertical residuals and the model is refitted until the mask is stable. This ensures that every reported inlier satisfies $|q_i-\hat{q}_i|\leq\tau$ for the line shown in the plot.

In [ ]:
TAU_GRID_MMOL_G = np.array([0.10, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 1.75, 2.00, 2.25,2.50, 2.75, 3.00, 4.00, 5.00, 6.00])


def format_tau(value):
    return f'{float(value):.2f}'.rstrip('0').rstrip('.')


def loo_q2(X, y):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    y = np.asarray(y, dtype=float).ravel()
    if len(y) <= X.shape[1] + 1:
        return -np.inf
    predictions = np.empty(len(y), dtype=float)
    for i in range(len(y)):
        train = np.arange(len(y)) != i
        model = LinearRegression().fit(X[train], y[train])
        predictions[i] = model.predict(X[i:i + 1])[0]
    denominator = np.sum((y - y.mean()) ** 2)
    if denominator <= np.finfo(float).eps:
        return -np.inf
    return float(1 - np.sum((y - predictions) ** 2) / denominator)


def stabilize_ransac_model(model, X, y, threshold, max_iterations=100):
    X_arr = np.asarray(X, dtype=float)
    y_arr = np.asarray(y, dtype=float).ravel()
    mask = np.asarray(model.inlier_mask_, dtype=bool).copy()
    seen = set()
    for _ in range(max_iterations):
        signature = mask.tobytes()
        if signature in seen or mask.sum() <= X_arr.shape[1]:
            return None
        seen.add(signature)
        estimator = LinearRegression().fit(X_arr[mask], y_arr[mask])
        updated_mask = np.abs(y_arr - estimator.predict(X_arr)) <= float(threshold)
        if np.array_equal(updated_mask, mask):
            model.estimator_ = estimator
            model.inlier_mask_ = mask
            return model
        mask = updated_mask
    return None


def optimize_ransac_hyperparameters(
    X, y, thresholds=None, min_samples=None, min_inliers=None,
    min_train_r2=0.60, min_cv_r2=0.50, max_trials=2000,
    stop_probability=0.999, require_positive=False, random_state=42
):
    X_arr = np.asarray(X, dtype=float)
    if X_arr.ndim == 1:
        X_arr = X_arr.reshape(-1, 1)
    y_arr = np.asarray(y, dtype=float).ravel()
    valid = np.isfinite(X_arr).all(axis=1) & np.isfinite(y_arr)
    X_arr = X_arr[valid]
    y_arr = y_arr[valid]
    n_samples, n_features = X_arr.shape
    min_inliers = max(n_features + 3, int(np.ceil(0.20 * n_samples))) if min_inliers is None else int(min_inliers)
    if n_samples < min_inliers:
        return None
    threshold_grid = TAU_GRID_MMOL_G if thresholds is None else np.atleast_1d(thresholds).astype(float)
    if min_samples is None:
        min_samples_grid = range(n_features + 1, n_features + 4)
    else:
        min_samples_grid = [int(min_samples)]
    best = None
    for current_min_samples in min_samples_grid:
        if current_min_samples >= n_samples:
            continue
        for current_threshold in threshold_grid:
            try:
                model = RANSACRegressor(
                    estimator=LinearRegression(),
                    min_samples=current_min_samples,
                    residual_threshold=float(current_threshold),
                    max_trials=max_trials,
                    stop_probability=stop_probability,
                    loss='absolute_error',
                    random_state=random_state
                ).fit(X_arr, y_arr)
            except ValueError:
                continue
            model = stabilize_ransac_model(
                model, X_arr, y_arr, float(current_threshold)
            )
            if model is None:
                continue
            mask = model.inlier_mask_
            n_inliers = int(mask.sum())
            if n_inliers < min_inliers:
                continue
            coefficients = np.asarray(model.estimator_.coef_, dtype=float)
            if require_positive and (coefficients.size != 1 or coefficients[0] <= 0):
                continue
            r2 = float(model.estimator_.score(X_arr[mask], y_arr[mask]))
            q2 = loo_q2(X_arr[mask], y_arr[mask])
            if not np.isfinite(q2) or r2 < min_train_r2 or q2 < min_cv_r2:
                continue
            coverage = n_inliers / n_samples
            score = q2 * coverage
            ranking = (score, coverage, q2, r2, -float(current_threshold), -current_min_samples)
            if best is None or ranking > best['ranking']:
                best = {
                    'model': model,
                    'threshold': float(current_threshold),
                    'min_samples': int(current_min_samples),
                    'min_inliers': min_inliers,
                    'n_inliers': n_inliers,
                    'coverage': coverage,
                    'r2': r2,
                    'q2': q2,
                    'score': score,
                    'ranking': ranking
                }
    return best


def ransac_plot(X, y, threshold, n, m, size_inliers, size_outliers, legend_fontsize, min_samples=None, min_inliers=4):
    X_arr = np.asarray(X, dtype=float).reshape(-1, 1)
    y_arr = np.asarray(y, dtype=float).ravel()
    try:
        current_ax = ax[n, m]
    except (TypeError, IndexError):
        current_ax = ax[m]
    tuned = optimize_ransac_hyperparameters(X_arr, y_arr, thresholds=None if threshold is None else [threshold], min_samples=min_samples, min_inliers=min_inliers)
    if tuned is None:
        current_ax.text(0.5, 0.5, 'No validated line', transform=current_ax.transAxes, ha='center', va='center')
        return None
    model = tuned['model']
    mask = model.inlier_mask_
    x_line = np.linspace(X_arr.min(), X_arr.max(), 200).reshape(-1, 1)
    coefficient = float(model.estimator_.coef_[0])
    intercept = float(model.estimator_.intercept_)
    current_ax.plot(
        x_line.ravel(), model.predict(x_line), color='black', linestyle='--',
        label=(f'Fit: y = {coefficient:.3f}x {intercept:+.2f}' + chr(10)
               + f'$R^2_{{in}}$={tuned["r2"]:.3f}, $Q^2_{{LOO}}$={tuned["q2"]:.3f}' + chr(10)
               + rf'$\tau$={format_tau(tuned["threshold"])}')
    )
    current_ax.scatter(X_arr[mask], y_arr[mask], facecolors='none', edgecolors='k', s=size_inliers, linewidths=1.2, label='Inliers')
    current_ax.legend(fontsize=legend_fontsize, loc='best')
    return tuned


### Regression settings

- descriptor: $X$;
- sorption: $Y$;
- rounded $\tau$ grid: 0.25–3.00 mmol/g;
- `max_trials = 2000` with `stop_probability = 0.999`;
- at least four inliers for one-dimensional lines and five for planes;
- candidate selection uses both predictive stability ($Q^2_{LOO}$) and inlier coverage.


### 7.1. Sequential RANSAC plots

In [ ]:
def sequential_ransac(
    X, y, threshold=None, sizes=None, n_iterations=20000, ax=None,
    min_r2=0.50, max_trials=20000, stop_probability=0.999,
    min_samples=None, min_cv_r2=0.50,
    markers=('o', 's', '^', 'D', 'P', 'X', 'v', '<', '>'),
    colors=None, legend_fontsize=10, min_inliers=4, text_offset=None,
    require_positive=False
):
    X_arr = np.asarray(X, dtype=float).reshape(-1, 1)
    y_arr = np.asarray(y, dtype=float).ravel()
    remaining_idx = np.arange(len(y_arr))
    models = []
    sizes = [80] * len(markers) if sizes is None else sizes
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color'] if colors is None else colors
    y_range = np.ptp(y_arr) if np.ptp(y_arr) != 0 else 1.0
    text_offset_value = (0.025 if text_offset is None else text_offset) * y_range

    for iteration in range(n_iterations):
        if len(remaining_idx) < min_inliers:
            break
        tuned = optimize_ransac_hyperparameters(
            X_arr[remaining_idx], y_arr[remaining_idx],
            thresholds=None if threshold is None else [threshold],
            min_samples=min_samples, min_inliers=min_inliers,
            min_train_r2=min_r2, min_cv_r2=min_cv_r2,
            max_trials=max_trials, stop_probability=stop_probability,
            require_positive=require_positive
        )
        if tuned is None:
            break
        model = tuned['model']
        inliers_idx = remaining_idx[model.inlier_mask_]
        coefficient = float(model.estimator_.coef_[0])
        x_line = np.linspace(X_arr.min(), X_arr.max(), 200).reshape(-1, 1)
        color = colors[iteration % len(colors)]
        marker = markers[iteration % len(markers)]
        ax.plot(
            x_line.ravel(), model.predict(x_line), linestyle='--', linewidth=2,
            color=color,
            label=(f'$slope$={coefficient:.3f}, $R^2_{{in}}$={tuned["r2"]:.2f}, '
                   + f'$Q^2_{{LOO}}$={tuned["q2"]:.2f}' + chr(10)
                   + rf'$\tau$={format_tau(tuned["threshold"])}')
        )
        ax.scatter(
            X_arr[inliers_idx].ravel(), y_arr[inliers_idx],
            marker=marker, s=sizes[iteration % len(sizes)], color=color,
            edgecolors='black', linewidths=1.2,
            label=f'$n_{{inliers}}$={len(inliers_idx)}'
        )
        models.append({
            'model': model,
            'inliers_idx': inliers_idx.copy(),
            'r2': tuned['r2'],
            'q2': tuned['q2'],
            'score': tuned['score'],
            'slope': coefficient,
            'intercept': float(model.estimator_.intercept_),
            'threshold': tuned['threshold'],
            'min_samples': tuned['min_samples'],
            'min_inliers': tuned['min_inliers']
        })
        remaining_idx = remaining_idx[~model.inlier_mask_]

    all_inliers = sorted(
        {int(index) for model in models for index in model['inliers_idx']},
        key=lambda index: (y_arr[index], index)
    )
    rank_map = {index: rank for rank, index in enumerate(all_inliers, start=1)}
    for model in models:
        model['ranks'] = [rank_map[int(index)] for index in model['inliers_idx']]
    for index in all_inliers:
        ax.text(
            float(X_arr[index].item()), float(y_arr[index] + text_offset_value),
            str(rank_map[index]), fontsize=9, ha='center', va='bottom', color='black'
        )
    if models:
        ax.legend(fontsize=legend_fontsize, loc='best')
    return models


def descriptor_tables(data, descriptor, models):
    inlier_rows = []
    inlier_indices = set()
    for model in models:
        q_corr = model['r2'] * len(model['inliers_idx']) / len(data)
        for index, rank in zip(model['inliers_idx'], model['ranks']):
            inlier_indices.add(int(index))
            inlier_rows.append({
                'Line': len({row['Line'] for row in inlier_rows}) + 1 if False else models.index(model) + 1,
                'Plot_Number': int(rank),
                'Name': data.iloc[index]['Name'],
                'Sorption': data.iloc[index]['Sorption, mmol/g'],
                descriptor: data.iloc[index][descriptor],
                'Slope': round(model['slope'], 3),
                'Intercept': round(model['intercept'], 3),
                'R2': round(model['r2'], 3),
                'Q2_LOO': round(model['q2'], 3),
                'n_inliers': len(model['inliers_idx']),
                'Inlier_ratio': round(len(model['inliers_idx']) / len(data), 3),
                'Residual_threshold': model['threshold'],
                'Min_samples': model['min_samples'],
                'Q_corr': round(q_corr, 3),
                'C_corr': round(1 - q_corr, 3)
            })
    outlier_indices = [index for index in range(len(data)) if index not in inlier_indices]
    outlier_indices = sorted(outlier_indices, key=lambda index: (data.iloc[index]['Sorption, mmol/g'], index))
    outlier_rows = [
        {
            'Outlier_Number': rank,
            'Name': data.iloc[index]['Name'],
            'Sorption': data.iloc[index]['Sorption, mmol/g'],
            descriptor: data.iloc[index][descriptor]
        }
        for rank, index in enumerate(outlier_indices, start=1)
    ]
    inliers = pd.DataFrame(inlier_rows).sort_values('Plot_Number').reset_index(drop=True) if inlier_rows else pd.DataFrame()
    outliers = pd.DataFrame(outlier_rows)
    return inliers, outliers


def analyze_descriptor(source_df, descriptor, label=None, min_inliers=None, n_iterations=5, dpi=600, require_positive=False):
    label = descriptor if label is None else label
    data = source_df[['Name', 'Sorption, mmol/g', descriptor]].copy()
    data[descriptor] = pd.to_numeric(data[descriptor].replace('—', np.nan), errors='coerce')
    data['Sorption, mmol/g'] = pd.to_numeric(data['Sorption, mmol/g'], errors='coerce')
    data = data.dropna().sort_values('Sorption, mmol/g', kind='mergesort').reset_index(drop=True)
    min_inliers = max(4, int(np.ceil(0.20 * len(data)))) if min_inliers is None else min_inliers
    fig, current_ax = plt.subplots(figsize=(10, 8), dpi=dpi)
    sns.scatterplot(
        ax=current_ax, x=data[descriptor], y=data['Sorption, mmol/g'],
        s=40, edgecolors='black', color='grey'
    )
    models = sequential_ransac(
        data[descriptor], data['Sorption, mmol/g'], threshold=None,
        sizes=[80] * 9, n_iterations=n_iterations, ax=current_ax,
        legend_fontsize=11, min_inliers=min_inliers, require_positive=require_positive
    )
    current_ax.set_xlabel(label, fontsize=14)
    current_ax.set_ylabel('Sorption (mmol/g)', fontsize=14)
    current_ax.set_title(f'Dependence of sorption on {label}')
    plt.tight_layout()
    plt.show()
    inliers, outliers = descriptor_tables(data, descriptor, models)
    print('Inliers')
    display(inliers)
    print('Outliers')
    display(outliers)
    return models, inliers, outliers


def analyze_cluster_groups(cluster_table, method_name):
    clusters = [value for value in sorted(cluster_table['Cluster'].astype(str).unique()) if value != 'Noise']
    ncols = min(3, len(clusters))
    nrows = int(np.ceil(len(clusters) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 7 * nrows), dpi=600)
    axes = np.atleast_1d(axes).ravel()
    inlier_reports = []
    outlier_reports = []
    for current_ax, cluster in zip(axes, clusters):
        data = cluster_table.loc[cluster_table['Cluster'].astype(str) == cluster].copy()
        data = data.sort_values('Sorption (mmol/g)', kind='mergesort').reset_index(drop=True)
        sns.scatterplot(
            ax=current_ax, x=data['ε (Dielectric constant)'], y=data['Sorption (mmol/g)'],
            s=40, edgecolors='black', color='grey'
        )
        models = sequential_ransac(
            data['ε (Dielectric constant)'], data['Sorption (mmol/g)'],
            sizes=[80] * 9, n_iterations=5, ax=current_ax,
            legend_fontsize=9, min_inliers=max(4, int(np.ceil(0.20 * len(data)))), require_positive=True
        )
        current_ax.set_title(f'{method_name}: Cluster {cluster}')
        current_ax.set_xlabel('ε (Dielectric constant)')
        current_ax.set_ylabel('Sorption (mmol/g)')
        report_data = data.rename(columns={'Sorption (mmol/g)': 'Sorption, mmol/g'})
        inliers, outliers = descriptor_tables(report_data, 'ε (Dielectric constant)', models)
        if not inliers.empty:
            inliers.insert(0, 'Cluster', cluster)
            inlier_reports.append(inliers)
        if not outliers.empty:
            outliers.insert(0, 'Cluster', cluster)
            outlier_reports.append(outliers)
    for current_ax in axes[len(clusters):]:
        current_ax.axis('off')
    plt.suptitle(f'{method_name} clusters with Sequential RANSAC')
    plt.tight_layout()
    plt.show()
    inlier_report = pd.concat(inlier_reports, ignore_index=True) if inlier_reports else pd.DataFrame()
    outlier_report = pd.concat(outlier_reports, ignore_index=True) if outlier_reports else pd.DataFrame()
    print('Inliers')
    display(inlier_report)
    print('Outliers')
    display(outlier_report)
    return inlier_report, outlier_report


In [ ]:
kmeans_inliers, kmeans_outliers = analyze_cluster_groups(df_kmeans, 'K-Means')


In [ ]:
hierarchical_inliers, hierarchical_outliers = analyze_cluster_groups(df_hc, 'Hierarchical')


In [ ]:
spectral_inliers, spectral_outliers = analyze_cluster_groups(df_spectral, 'Spectral')


In [ ]:
dbscan_inliers, dbscan_outliers = analyze_cluster_groups(df_dbscan, 'DBSCAN')


## 8. Spectral Clustering tuning

It was mentioned earlier that spectral clustering gives unsatisfactory clustering results. This conclusion can be drawn from the fact that after choosing the standard sigma value in the adjacency matrix, its values were mostly zero, which indicates the dissimilarity of points in the feature space, which leads us to an abnormal number of clusters that do not make sense. The standardization of data for finding optimal values of the adjacency matrix is illustrated below. If most of the values turn out to be non-zero or one, then the spectral clustering model can be revised, but in our case this did not happen.

In [ ]:
X_z = StandardScaler().fit_transform(X)
distance_matrix_z = pdist(X_z, metric='euclidean')
sigma_z = np.median(distance_matrix_z)
A_z = -np.square(X_z[:, None] - X_z[None, :]).sum(axis=-1)
A_z = np.exp(A_z / (2 * sigma_z ** 2))
np.fill_diagonal(A_z, 0)


In [ ]:
sns.set(font_scale=1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

bin_values, _, _ = ax1.hist(A_z.flatten(), bins=50)

ax2.hist(A_z.flatten(), bins=50)
ax2.set_ylim([0, bin_values[9]])
ax1.set_xlabel('Value')
ax2.set_xlabel('Value')
ax1.set_ylabel('Frequency')
ax1.set_title('Adjacency Matrix Values')
ax2.set_title('Area of small values (first 50 bins)')
fig.suptitle('Adjacency Matrix Value Distribution tuned by median heuristic (Standart Scaler)')
plt.show()


In [ ]:
X_rob = RobustScaler().fit_transform(X)
distance_matrix_rob = pdist(X_rob, metric='euclidean')
sigma_rob = np.median(distance_matrix_rob)
A_rob = -np.square(X_rob[:, None] - X_rob[None, :]).sum(axis=-1)
A_rob = np.exp(A_rob / (2 * sigma_rob ** 2))
np.fill_diagonal(A_rob, 0)


In [ ]:
sns.set(font_scale=1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

bin_values, _, _ = ax1.hist(A_rob.flatten(), bins=50)

ax2.hist(A_rob.flatten(), bins=50)
ax2.set_ylim([0, bin_values[9]])
ax1.set_xlabel('Value')
ax2.set_xlabel('Value')
ax1.set_ylabel('Frequency')
ax1.set_title('Adjacency Matrix Values')
ax2.set_title('Area of small values (first 50 bins)')
fig.suptitle('Adjacency Matrix Value Distribution tuned by median heuristic (Robust Scaler)')
plt.show()


## 10. Additional research: Kamlet-Taft correlations

In [ ]:
df = df.copy()
df.sample(5)


In [ ]:
df_sorted = df.iloc[:, :7].dropna(axis=0)
df_sorted


In [ ]:
def correlation_loss(r2_in, n_inliers, n_total):
    q_corr = r2_in * (n_inliers / n_total)
    c_corr = 1 - q_corr
    return q_corr, c_corr


In [ ]:
sns.set(font_scale=1.7)
fig, axes = plt.subplots(1, 2, figsize=(16, 10), dpi=600)
models_kamlet_taft = {}
data_kamlet_taft = {}

for current_ax, parameter in zip(axes, ['β', 'π*']):
    data = df[['Name', 'Sorption, mmol/g', parameter]].copy()
    data[parameter] = pd.to_numeric(data[parameter].replace('—', np.nan), errors='coerce')
    data = data.dropna().sort_values('Sorption, mmol/g', kind='mergesort').reset_index(drop=True)
    sns.scatterplot(
        ax=current_ax, x=data[parameter], y=data['Sorption, mmol/g'],
        s=40, edgecolors='black', color='grey'
    )
    models = sequential_ransac(
        data[parameter], data['Sorption, mmol/g'], threshold=None,
        sizes=[80] * 9, n_iterations=5, ax=current_ax,
        legend_fontsize=12, min_inliers=max(4, int(np.ceil(0.20 * len(data)))), require_positive=True
    )
    current_ax.set_xlabel(parameter, fontsize=14)
    current_ax.set_ylabel('Sorption (mmol/g)', fontsize=14)
    models_kamlet_taft[parameter] = models
    data_kamlet_taft[parameter] = data

plt.suptitle('Dependence of sorption on Kamlet-Taft parameters')
plt.tight_layout()
plt.show()

for parameter in ['β', 'π*']:
    inliers, outliers = descriptor_tables(data_kamlet_taft[parameter], parameter, models_kamlet_taft[parameter])
    print(f'{parameter}: inliers')
    display(inliers)
    print(f'{parameter}: outliers')
    display(outliers)

models_beta = models_kamlet_taft['β']
models_pi_star = models_kamlet_taft['π*']


### Correlation reporting

For every accepted line, the notebook reports $R^2_{in}$, $Q^2_{LOO}$, the number and fraction of inliers, the selected rounded threshold $\tau$, and `min_samples`.

The original descriptive score is retained for continuity:

$$
Q_{corr} = R^2_{in}\frac{N_{in}}{N_{total}},
\qquad
C_{corr} = 1-Q_{corr}.
$$

$Q_{corr}$ summarizes training linearity and coverage, whereas $Q^2_{LOO}$ checks how stable the fitted inlier line is when each inlier is predicted in turn without being used for that prediction.


In [ ]:
models_alpha, inliers_alpha, outliers_alpha = analyze_descriptor(df, 'α', 'Kamlet-Taft α', dpi=600, require_positive=True)


In [ ]:
models_dn, inliers_dn, outliers_dn = analyze_descriptor(df, 'DN', 'Donor Number (DN)', dpi=600, require_positive=True)


## 11. Donor number correlations

In [ ]:


def ransac_plane_labels(df, x_col, y_col, z_col='Sorption, mmol/g', min_inliers=5):
    data = df[['Name', x_col, y_col, z_col]].dropna().sort_values(z_col, kind='mergesort').reset_index(drop=True)
    X_plane = data[[x_col, y_col]].to_numpy(float)
    z = data[z_col].to_numpy(float)
    tuned = optimize_ransac_hyperparameters(X_plane, z, min_inliers=min_inliers)
    if tuned is None:
        raise ValueError(f'No validated plane for {x_col} and {y_col}')
    model = tuned['model']
    mask = model.inlier_mask_
    a, b = model.estimator_.coef_
    c = model.estimator_.intercept_
    x = X_plane[:, 0]
    y = X_plane[:, 1]
    fig = plt.figure(figsize=(8, 10), dpi=600)
    current_ax = fig.add_subplot(111, projection='3d')
    current_ax.scatter(x, y, z, s=30, c='lightgray', edgecolors='k', linewidths=0.35)
    current_ax.scatter(x[mask], y[mask], z[mask], s=55, c='royalblue', edgecolors='k', linewidths=0.8, label='inliers')
    x_grid = np.linspace(x.min(), x.max(), 25)
    y_grid = np.linspace(y.min(), y.max(), 25)
    xx, yy = np.meshgrid(x_grid, y_grid)
    zz = a * xx + b * yy + c
    current_ax.plot_surface(xx, yy, zz, alpha=0.28, linewidth=0, antialiased=True, cmap='viridis')
    current_ax.set_xlabel(x_col)
    current_ax.set_ylabel(y_col)
    current_ax.set_zlabel(z_col)
    current_ax.set_title(
        rf'$S={a:.2f}{x_col}{b:+.2f}{y_col}{c:+.2f}$' + chr(10) +
        rf'$R^2_{{in}}={tuned["r2"]:.2f}, Q^2_{{LOO}}={tuned["q2"]:.2f}, \tau={format_tau(tuned["threshold"])}$'
    )
    for rank, index in enumerate(np.flatnonzero(mask), start=1):
        current_ax.text(x[index], y[index], z[index], str(rank), fontsize=8)
    current_ax.legend(loc='best')
    plt.tight_layout()
    plt.show()
    table = data.copy()
    table['Predicted_sorption'] = model.predict(X_plane)
    table['Residual'] = z - table['Predicted_sorption']
    inlier_table = table.loc[mask].copy().reset_index(drop=True)
    inlier_table.insert(0, 'Plot_Number', np.arange(1, len(inlier_table) + 1))
    outlier_table = table.loc[~mask].copy().sort_values(z_col, kind='mergesort').reset_index(drop=True)
    outlier_table.insert(0, 'Outlier_Number', np.arange(1, len(outlier_table) + 1))
    print('Inliers')
    display(inlier_table)
    print('Outliers')
    display(outlier_table)
    return model, inlier_table, outlier_table


In [ ]:
model_bp, inliers_bp, outliers_bp = ransac_plane_labels(df_sorted, 'β', 'π*')


In [ ]:
model_bdn, inliers_bdn, outliers_bdn = ransac_plane_labels(df_sorted, 'β', 'DN')


In [ ]:
model_pidn, inliers_pidn, outliers_pidn = ransac_plane_labels(df_sorted, 'π*', 'DN')


## 12. Conclusions
Reassess conclusions after restarting the kernel and running all cells on the filtered dataset. Earlier conclusions referred to the unfiltered sample.


## 13. Additional molecular descriptors

The following sections preserve the same plot structure: all compounds are shown in grey, while only validated inliers are coloured and numbered. Outliers are reported only in the table below each graph.


In [ ]:
tpsa_2d_col = next(column for column in df.columns if '2D' in column.upper() and 'PSA' in column.upper())
tpsa_3d_col = next(column for column in df.columns if '3D' in column.upper() and 'TPSA' in column.upper())
polar_sasa_col = next(column for column in df.columns if 'POLAR SASA 3D' in column.upper())


### Proton affinity


In [ ]:
models_pa, inliers_pa, outliers_pa = analyze_descriptor(df, 'PA, kJ/mol', 'Proton Affinity (kJ/mol)')


### Topological polar surface area


In [ ]:
models_tpsa2d, inliers_tpsa2d, outliers_tpsa2d = analyze_descriptor(df, tpsa_2d_col, 'TopoPSA 2D (Å²)')


In [ ]:
models_tpsa3d, inliers_tpsa3d, outliers_tpsa3d = analyze_descriptor(df, tpsa_3d_col, 'TPSA 3D (Mordred CPSA, Å²)')


### Molecular cross-sections


In [ ]:
models_sigma_eb, inliers_sigma_eb, outliers_sigma_eb = analyze_descriptor(df, 'σEB, Å²', 'Molecular cross-section σEB (Å²)')


In [ ]:
models_sigma_w, inliers_sigma_w, outliers_sigma_w = analyze_descriptor(df, 'σW, Å²', 'Molecular cross-section σW (Å²)')


### Three-dimensional polar SASA


In [ ]:
models_polar_sasa, inliers_polar_sasa, outliers_polar_sasa = analyze_descriptor(df, polar_sasa_col, 'Polar SASA 3D (Å²)')


## 14. Collinearity diagnostics

Pairwise correlations, variance inflation factors, and PCA are calculated for the five Kamlet–Taft-related predictors before the multivariate model is interpreted.


In [ ]:
KT_PREDICTORS = ['ε', 'α', 'β', 'π*', 'DN']
collinearity_data = df[['Name', 'Sorption, mmol/g'] + KT_PREDICTORS].copy()
for column in KT_PREDICTORS + ['Sorption, mmol/g']:
    collinearity_data[column] = pd.to_numeric(collinearity_data[column].replace('—', np.nan), errors='coerce')
collinearity_data = collinearity_data.dropna().reset_index(drop=True)
X_collinearity = collinearity_data[KT_PREDICTORS].to_numpy(float)
X_collinearity_scaled = StandardScaler().fit_transform(X_collinearity)

correlation_matrix = collinearity_data[KT_PREDICTORS].corr()
vif_rows = []
for index, descriptor in enumerate(KT_PREDICTORS):
    target = X_collinearity_scaled[:, index]
    predictors = np.delete(X_collinearity_scaled, index, axis=1)
    r2 = LinearRegression().fit(predictors, target).score(predictors, target)
    vif_rows.append({'Descriptor': descriptor, 'VIF': np.inf if 1 - r2 <= 1e-12 else 1 / (1 - r2)})
vif_table = pd.DataFrame(vif_rows).sort_values('VIF', ascending=False).reset_index(drop=True)

pca = PCA().fit(X_collinearity_scaled)
components = np.arange(1, len(KT_PREDICTORS) + 1)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True, ax=axes[0])
axes[0].set_title('Descriptor correlation matrix')
axes[1].bar(components, pca.explained_variance_ratio_)
axes[1].plot(components, np.cumsum(pca.explained_variance_ratio_), marker='o')
axes[1].set_xticks(components)
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel('Principal component')
axes[1].set_ylabel('Explained variance')
axes[1].set_title('PCA explained variance')
plt.tight_layout()
plt.show()

display(vif_table.round(3))
display(pd.DataFrame(pca.components_.T, index=KT_PREDICTORS, columns=[f'PC{i}' for i in components]).round(3))


## 15. Sequential RANSAC permutation test

Sorption is permuted relative to each descriptor 1000 times. The observed Sequential RANSAC settings are kept fixed, and the full sequential extraction is repeated for every shuffled response. The statistic is $S_{corr}=\sum_i Q^2_{LOO,i}N_{in,i}/N$.
 For every permuted fit, the inlier mask is stabilized against the final refitted line using the same rule as in the observed data.

In [ ]:
def fixed_sequential_score(X, y, fitted_models, min_inliers, seed, max_trials=300):
    X = np.asarray(X, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=float).ravel()
    remaining = np.arange(len(y))
    total = len(y)
    score = 0.0
    for line, fitted in enumerate(fitted_models, start=1):
        if len(remaining) < max(min_inliers, fitted['min_samples'] + 1):
            break
        try:
            model = RANSACRegressor(
                estimator=LinearRegression(), min_samples=fitted['min_samples'],
                residual_threshold=fitted['threshold'], max_trials=max_trials,
                stop_probability=0.999, loss='absolute_error', random_state=seed + line
            ).fit(X[remaining], y[remaining])
        except ValueError:
            break
        model = stabilize_ransac_model(
            model, X[remaining], y[remaining], fitted['threshold']
        )
        if model is None:
            break
        mask = model.inlier_mask_
        if mask.sum() < min_inliers:
            break
        r2 = model.estimator_.score(X[remaining][mask], y[remaining][mask])
        q2 = loo_q2(X[remaining][mask], y[remaining][mask])
        if not np.isfinite(q2) or r2 < 0.60 or q2 < 0.50:
            break
        score += q2 * mask.sum() / total
        remaining = remaining[~mask]
    return float(score)


def permutation_test_descriptor(source_df, descriptor, fitted_models, n_permutations=1000):
    data = source_df[[descriptor, 'Sorption, mmol/g']].copy()
    data[descriptor] = pd.to_numeric(data[descriptor].replace('—', np.nan), errors='coerce')
    data = data.dropna().reset_index(drop=True)
    X_descriptor = data[descriptor].to_numpy(float)
    y_sorption = data['Sorption, mmol/g'].to_numpy(float)
    observed = sum(model['q2'] * len(model['inliers_idx']) / len(data) for model in fitted_models)
    min_inliers = max(4, int(np.ceil(0.20 * len(data))))
    seeds = np.random.default_rng(42).integers(0, 2 ** 32 - 1, size=n_permutations)
    null = np.asarray(Parallel(n_jobs=-1)(
        delayed(fixed_sequential_score)(
            X_descriptor,
            np.random.default_rng(int(seed)).permutation(y_sorption),
            fitted_models,
            min_inliers,
            int(seed)
        )
        for seed in seeds
    ))
    p_value = (1 + np.sum(null >= observed)) / (n_permutations + 1)
    return observed, null, p_value


permutation_models = {
    'α': models_alpha,
    'β': models_beta,
    'π*': models_pi_star,
    'DN': models_dn
}
permutation_results = {}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for current_ax, (descriptor, fitted_models) in zip(axes.ravel(), permutation_models.items()):
    observed, null, p_value = permutation_test_descriptor(df, descriptor, fitted_models, n_permutations=1000)
    permutation_results[descriptor] = {
        'Descriptor': descriptor,
        'Observed_S_corr': observed,
        'Null_mean': null.mean(),
        'Null_95_percentile': np.quantile(null, 0.95),
        'Permutation_p': p_value
    }
    current_ax.hist(null, bins=30)
    current_ax.axvline(observed, linestyle='--', linewidth=2, color='black')
    current_ax.set_title(f'{descriptor}: p={p_value:.4f}')
    current_ax.set_xlabel(r'$S_{\mathrm{corr}}$ under permutation')
plt.tight_layout()
plt.show()
display(pd.DataFrame(permutation_results.values()).round(4))


## 16. Nested leave-one-out cross-validation

PLS is used because correlated predictors are expected. Each compound is held out once, while the number of PLS components is chosen only inside the corresponding training fold.


In [ ]:
X_pls = collinearity_data[KT_PREDICTORS].to_numpy(float)
y_pls = collinearity_data['Sorption, mmol/g'].to_numpy(float)
predictions = np.empty(len(y_pls))
selected_components = np.empty(len(y_pls), dtype=int)

for fold, (train_index, test_index) in enumerate(LeaveOneOut().split(X_pls)):
    maximum_components = min(len(KT_PREDICTORS), len(train_index) - 1)
    inner_cv = KFold(n_splits=min(5, len(train_index)), shuffle=True, random_state=42 + fold)
    pipeline = Pipeline([
        ('scale', StandardScaler()),
        ('pls', PLSRegression(scale=False, max_iter=2000))
    ])
    search = GridSearchCV(
        pipeline,
        {'pls__n_components': list(range(1, maximum_components + 1))},
        scoring='neg_mean_squared_error',
        cv=inner_cv,
        n_jobs=-1
    )
    search.fit(X_pls[train_index], y_pls[train_index])
    predictions[test_index[0]] = np.ravel(search.predict(X_pls[test_index]))[0]
    selected_components[test_index[0]] = search.best_params_['pls__n_components']

residuals = y_pls - predictions
q2_nested = 1 - np.sum(residuals ** 2) / np.sum((y_pls - y_pls.mean()) ** 2)
rmsecv = np.sqrt(np.mean(residuals ** 2))
maecv = np.mean(np.abs(residuals))

fig, current_ax = plt.subplots(figsize=(8, 7))
current_ax.scatter(y_pls, predictions, s=50, edgecolors='black')
limits = [min(y_pls.min(), predictions.min()), max(y_pls.max(), predictions.max())]
current_ax.plot(limits, limits, '--', color='black')
for rank, index in enumerate(np.argsort(y_pls), start=1):
    current_ax.text(y_pls[index], predictions[index], str(rank), fontsize=9, ha='center', va='bottom')
current_ax.set_xlabel('Observed sorption (mmol/g)')
current_ax.set_ylabel('LOO-predicted sorption (mmol/g)')
current_ax.set_title(f'Nested LOO PLS: Q²={q2_nested:.3f}, RMSECV={rmsecv:.3f}')
plt.tight_layout()
plt.show()

cv_metrics = pd.DataFrame({
    'Complete_cases': [len(y_pls)],
    'Predictors': [len(KT_PREDICTORS)],
    'Q2_LOO_nested': [q2_nested],
    'RMSECV': [rmsecv],
    'MAECV': [maecv]
})
cv_table = collinearity_data[['Name', 'Sorption, mmol/g'] + KT_PREDICTORS].copy()
cv_table['Predicted_sorption'] = predictions
cv_table['Residual'] = residuals
cv_table['Selected_PLS_components'] = selected_components
display(cv_metrics.round(4))
display(cv_table.round(3))
